In [11]:
### this is the main inference file used for website

## preprocessing
from flask import Flask, render_template, request, send_file
#%load_ext autoreload
#%autoreload 2
import os
import random
import numpy as np
import pandas as pd
import torch
import sys
import io
import matplotlib as plt
import time
from tqdm import tqdm
current_pth = os.path.abspath('../')
#print(current_pth)
sys.path.append(current_pth)
from utils.sequence_process import SeqProcessConfig, HaplotypeSeqProcessor,validate_df
from utils.data_preprocess import  drop_nan_missing_values,drop_wilde_type,renormalize
from utils.data_preprocess import transform_genseq_upper
from utils.utilities import ReaderWriter, report_available_cuda_devices,get_device
from utils.data_preprocess import create_directory
from proportion_model.src.predict_model import BEDICT_EncEnc_HaplotypeModel

## for absolute efficiency model
working_path = '../absolute_efficiency_model'
sys.path.append(working_path)
from src.utils import one_hot_encode,compute_eval_results_df
from models.dataset import ProtospacerDataset
from models.data_process import generate_partition_datatensor,get_datatensor_partitions_for_inference,prepare_sample_data
from models.trainval_workflow import run_inference
import re
from flask import jsonify

In [10]:
!pip install prettytable

  Obtaining dependency information for prettytable from https://files.pythonhosted.org/packages/4d/81/316b6a55a0d1f327d04cc7b0ba9d04058cb62de6c3a4d4b0df280cbe3b0b/prettytable-3.9.0-py3-none-any.whl.metadata


In [12]:
def get_overall_prob(pdf, edf, num_runs):
    
    df_new = pdf.copy()
    #print(df_new.columns)
    wild_type = []
    for i in range(num_runs):
        ID=pdf.iloc[0]['seq_id']
        pred_y = edf[edf['seq_id']== ID][f'pred_class_run_{i}'].tolist()
        x_pred = pdf[pdf['seq_id']==ID][f'score_run_{i}']
        x_pred = np.array(x_pred)
        df_new[f'overall_pred_run_{i}'] = x_pred*pred_y[0]
        ## adding back the wild type
        wild_type.append(1- pred_y[0])
    
    list_1 = [ID, pdf.iloc[0]['Inp_seq'], pdf.iloc[0]['Inp_seq']]
    list_1.extend(wild_type)
    list_1.extend(wild_type)
    new_row = pd.Series(list_1, index=df_new.columns)
    #print(new_row)
    df_new.loc[len(df_new)] = new_row
    return df_new

In [13]:
def main(df,editor_name, in_vitro,lib_name, editing_window):
    
    #editor_name = 'ABE8e-NG'
    #in_vitro = True
    #lib_name =''
    
    input_type ='protospacer_PAM'
    num_runs = 3
    data_name = 'KM_SpCas9-ABEmax_v2'

    model_name= 'CNN'
    version = 2
    gpu_index = 0
    print(df)
    #data_pth = os.path.abspath('../../../crispr_private')
    #data_dir = create_directory(os.path.join(data_pth, 'dataset', 'final_dataset'))
    #df = pd.read_excel(os.path.join(data_dir,f'{data_name}.xlsx'),header=0)
    #df.rename(columns={'rname':'ID'}, inplace=True)
    #df.rename(columns={'refSeq':'Reference'}, inplace=True)
    #df.rename(columns={'seq':'Outcome'}, inplace=True)
    #df.rename(columns={'refseq_pam':'PAM'}, inplace=True)
    #df = transform_genseq_upper(df, ['refSeq_full','refseq_leftoverhang', 'Reference','PAM', 'refseq_rightoverhang' ])
    df = transform_genseq_upper(df, ['protospacer_PAM'])
    df['ID'] = df['protospacer_PAM']
    #protospacer_PAM =pd.DataFrame(df['Reference'] + df['PAM'])
    #protospacer_PAM.columns = ['protospacer_PAM']
    #extended_df = pd.concat([df,protospacer_PAM],axis=1)
    tseq_col = 'protospacer_PAM'
    #df = extended_df.copy()
    report_available_cuda_devices()
    device = get_device(True, 0)

    
    target_conv_nucl = {'ABEmax-NG':('A', 'G'),'ABE8e-NG':('A', 'G'), 'ABE8e-SpCas9':('A', 'G'), 'ABE8e-SpRY':('A', 'G'), 'ABEmax-SpCas9':('A','G'),'ABEmax-SpRY':('A', 'G'),}
    seqconfig = SeqProcessConfig(24, (1,24), (editing_window[0],editing_window[1]), 1)
    seq_processor = HaplotypeSeqProcessor(editor_name, target_conv_nucl[editor_name], seqconfig)
    bedict = BEDICT_EncEnc_HaplotypeModel(seq_processor, seqconfig, device)
    
    ## select the proportion model
    if in_vitro:
        model_path = current_pth +  f'/proportion_model/output/experiment_run_proportions_encenc_two_model/{editor_name}_proportions_encenc_two_model/'+ input_type + '/exp_version_0/train_val/'
        #proportion = []
    else:
        model_path = current_pth +  '/proportion_model/output/experiment_run_proportions_encenc_two_model/'+lib_name+ f'/{editor_name}_proportions_encenc_two_model/'+ input_type + '/exp_version_0/train_val/'

    print('we are loding the model from:',model_path)    
    proportion_df = pd.DataFrame(columns=['seq_id', 'Inp_seq', 'Outp_seq'])
    for i in range(num_runs):
    
        model_dir = model_path + f'run_{i}'
        start_test = time.time()
        #print(len(df))
        pred_df = bedict.predict_from_dataframe(df, ['ID','protospacer_PAM'] ,model_dir, outpseq_col=None, outcome_col=None, renormalize=True, batch_size=5)
        elapsed_time_test = time.time() - start_test
        #print(f"inference for {len(df)}: Time used = {elapsed_time_test:.4f} seconds")
        proportion_df['seq_id']=  pred_df['seq_id']
        proportion_df['Inp_seq']=  pred_df['Inp_seq']   
        proportion_df['Outp_seq']=  pred_df['Outp_seq'] 
        proportion_df[f'score_run_{i}']=  pred_df['pred_score'] 

    pd.set_option('display.float_format', '{:.6f}'.format)

    
    #absolute efficiency model prediciton
    x_protospacer, y,ID, x_non_protos_f = prepare_sample_data(df,input_type,'proportion')
    if in_vitro:
        model_path = os.path.join(current_pth, 'absolute_efficiency_model',
                                          'output', 
                                          f'{model_name}_v{version}',editor_name, 
                                         input_type)
    else:
        model_path = os.path.join(current_pth, 'absolute_efficiency_model',
                                          'output', 
                                          f'{model_name}_v{version}','invivo', lib_name, editor_name, 
                                         input_type)

    print('we are loading the model from:',model_path)
    data_partitions = {}
    for num_run in range(num_runs):
        data_partitions[num_run] = {'train_index': None, 'test_index':np.arange(x_protospacer.shape[0]) }

    if model_name == 'CNN':
        proc_x_protospacer = one_hot_encode(x_protospacer)
        proc_x_protospacer = proc_x_protospacer.reshape(proc_x_protospacer.shape[0], -1)

    dpartitions, datatensor_partitions = get_datatensor_partitions_for_inference(data_partitions,
                                                                   model_name,
                                                                   proc_x_protospacer,
                                                                   y,ID,
                                                                   x_non_protos_f,
                                                                   fdtype=torch.float32,
                                                                   train_size=0.0,
                                                                   random_state=42)

    
    train_val_path = os.path.join(model_path, 'train_val')
    test_path = os.path.join(model_path, 'sample_test', data_name)
    print(f'Running model: {model_name}, exp_name: {input_type}, saved at {train_val_path}')
    a, b = run_inference(datatensor_partitions, 
                                 train_val_path, 
                                 test_path, 
                                 gpu_index, 
                                 to_gpu=True)
                                 #num_runs=num_runs)
    print('='*15)
    
    #efficiency_df = pd.DataFrame(columns=['seq_id', 'Inp_seq', 'Outp_seq', 'true_class'])
    for i in range(num_runs):
        if i == 0:
            efficiency_df = pd.read_csv(os.path.join(test_path,'run_'f'{i}', 'predictions_test.csv'), )
            efficiency_df.rename(columns={'pred_class':f'pred_class_run_{i}'}, inplace=True)
            efficiency_df.rename(columns={'id':'seq_id'}, inplace=True)


        else:
            temp_df= pd.read_csv(os.path.join(test_path,'run_'f'{i}', 'predictions_test.csv'), )
            pred = temp_df['pred_class']
            efficiency_df[f'pred_class_run_{i}'] = pred


    
    dfg = proportion_df.groupby('seq_id')
    final_df = dfg.apply(get_overall_prob, efficiency_df,num_runs)
    final_df.reset_index(inplace=True, drop=True)
    columns_to_remove = [f'score_run_{i}' for i in range(num_runs)]
    final_df = final_df.drop(columns=columns_to_remove)
    selected_columns = [f'overall_pred_run_{i}' for i in range(num_runs)]
    final_df['average_overal_pred'] = final_df[selected_columns].mean(axis=1)
    final_df['std_overal_prd'] = final_df[selected_columns].std(axis=1)


    ranked_df = final_df.copy()
    ranked_df['rank'] = ranked_df.groupby('seq_id')['average_overal_pred'].rank(ascending=False)
    ranked_df['rank'] = ranked_df['rank'].astype(int)
    df_sorted = ranked_df.sort_values(by=['seq_id', 'rank'])
    ## rank the output
    
    ### display the result
    tseqids = df_sorted['seq_id'][0]
    tseqids = [tseqids]
    df_small = df_sorted[df_sorted['seq_id']==tseqids[0]].copy()
    
    res_html = bedict.visualize_haplotype(df= df_small, seqsids_lst= tseqids, 
                                            inpseq_cols = ['seq_id','Inp_seq'], 
                                            outpseq_col = 'Outp_seq', 
                                            outcome_col='average_overal_pred', 
                                            predscore_thr=0.)
    
    return final_df, df_sorted, res_html


In [14]:
# Set a seed for reproducibility
import random
import pandas as pd
random.seed(42)

# Number of rows in the DataFrame
num_rows = 10  # You can adjust this based on your needs

# Creating a DataFrame with a column 'protospacer_PAM'
df = pd.DataFrame({
    'protospacer_PAM': [''.join(random.choice('ACGT') for _ in range(24)) for _ in range(num_rows)]
})

# Display the DataFrame
print("Sample DataFrame:")
print(df)


Sample DataFrame:
            protospacer_PAM
0  AAGCCCAATAAACCACTCTGACTG
1  GCCGAATAGGGATATAGGCAACGA
2  CATGTGCGGCGACCCTTGCGACAG
3  TGACGCTTTCGCCGTTGCCTAAAC
4  CTATTTGAAGGAGTCTAGCAGCCG
5  CAGTAAGGCACAATACCTCGTCCG
6  TGTTACCAGACCAAACAAGACGTC
7  CTCTTCAATGTTTAAATGACCCTC
8  TCGTCATAAAACCTTTCTACTATG
9  TGTTCCGCAAGAATCAACAACTAC


In [15]:
in_vitro = True
lib_name = ''
editing_window = [1,20]
editor_name = 'ABE8e-NG'
main(df,editor_name, in_vitro,lib_name, editing_window)

            protospacer_PAM
0  AAGCCCAATAAACCACTCTGACTG
1  GCCGAATAGGGATATAGGCAACGA
2  CATGTGCGGCGACCCTTGCGACAG
3  TGACGCTTTCGCCGTTGCCTAAAC
4  CTATTTGAAGGAGTCTAGCAGCCG
5  CAGTAAGGCACAATACCTCGTCCG
6  TGTTACCAGACCAAACAAGACGTC
7  CTCTTCAATGTTTAAATGACCCTC
8  TCGTCATAAAACCTTTCTACTATG
9  TGTTCCGCAAGAATCAACAACTAC
no GPU devices available!!
+--------------------------------------+----------+
|             Description              |  Value   |
+--------------------------------------+----------+
|             Base editor              | ABE8e-NG |
|          Target nucleotide           |    A     |
|        Conversion nucleotide         |    G     |
| Maximum number of targets considered |    8     |
+--------------------------------------+----------+
+------------------------------------------------+-------+
|           Sequence processing Config           | Value |
+------------------------------------------------+-------+
|                sequence length                 |   24  |
|    sequence

225it [00:00, 18008.52it/s]                                                     


input sequence length: 24
number of NA: 0
--- generating edit combinations ---


10it [00:00, 179.75it/s]


comb_final_df:
 Index(['seq_id', 'Inp_seq', 'Outp_seq'], dtype='object')
24


...

input sequence length: 24


250it [00:00, 1156.94it/s]                                                      


number of NA: 0


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  7.16it/s]


(                        seq_id                   Inp_seq  \
 0     AAGCCCAATAAACCACTCTGACTG  AAGCCCAATAAACCACTCTGACTG   
 1     AAGCCCAATAAACCACTCTGACTG  AAGCCCAATAAACCACTCTGACTG   
 2     AAGCCCAATAAACCACTCTGACTG  AAGCCCAATAAACCACTCTGACTG   
 3     AAGCCCAATAAACCACTCTGACTG  AAGCCCAATAAACCACTCTGACTG   
 4     AAGCCCAATAAACCACTCTGACTG  AAGCCCAATAAACCACTCTGACTG   
 ...                        ...                       ...   
 1217  TGTTCCGCAAGAATCAACAACTAC  TGTTCCGCAAGAATCAACAACTAC   
 1218  TGTTCCGCAAGAATCAACAACTAC  TGTTCCGCAAGAATCAACAACTAC   
 1219  TGTTCCGCAAGAATCAACAACTAC  TGTTCCGCAAGAATCAACAACTAC   
 1220  TGTTCCGCAAGAATCAACAACTAC  TGTTCCGCAAGAATCAACAACTAC   
 1221  TGTTCCGCAAGAATCAACAACTAC  TGTTCCGCAAGAATCAACAACTAC   
 
                       Outp_seq  overall_pred_run_0  overall_pred_run_1  \
 0     AAGCCCAATAAACCGCTCTGACTG            0.027020            0.007337   
 1     AAGCCCAATAAGCCACTCTGACTG            0.027775            0.034504   
 2     AAGCCCAATAAGCCGCTCTGACTG          